## Before You Get Started Check-List

Before starting work on the competition, it’s recommended to go through the following steps to understand the setup, data, and submission expectations.

---

### 1. Read the Competition Page in Detail

Begin with the **Overview** and **Data** tabs on the competition page.

In particular, read the **Evaluation and Submission Guide** section carefully.

---

### 2. Review the Rules Page (Especially Sections 6–9)

Next, go through the **Rules** page, with special attention to **Sections 6, 7, 8, and 9**, which are especially relevant for this competition.

Key points to keep in mind:
- Submissions are expected to be **Notebook/Code Only**.
- Only the **competition-provided data** should be used.
- Each submission should include the **link to the exact notebook (version-sensitive)** used to generate it in the submission description.
- The notebook may be private, but it must be **shared with the competition host accounts**(host usernames-rahulsundar, sanchitbedi, siddharthandileep) so results can be reviewed and reproduced.

Reading the full rules is important to avoid issues during evaluation or shortlisting.

---

### 3. Go Through the Helper Notebook

The provided **helper notebook** is designed to guide you through the competition setup and data.

It contains instructions covering:
- Kaggle compute environment and runtime constraints
- Connecting and working with the provided GitHub repository
- Understanding the Data
- Creating train/validation time-series samples
- Test Input Format
- An overview of a complete end-to-end notebook pipeline
- Practical training tips and recommendations

It will help you get oriented before implementing your own approach.

---

### 4. Review the Baseline Repository

A baseline code repository is provided.

It’s useful to:
- Read the repository **README**
- Understand the expected pipeline and code organization

This step is optional, but can serve as a helpful reference when building your own solution.

---

### 5. Get Started

You’re ready now to start developing your own solution for the competition.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## ‼️ Add **'aisehack-theme-2'** competition dataset as an input and before running the cells

In [ ]:
# where all input datasets are stored
os.listdir('/kaggle/input/')

In [ ]:
# different input data folders provided
os.listdir('/kaggle/input/competitions/aisehack-theme-2')

In [ ]:
# current working directory
os.getcwd()

## 1. Kaggle Notebook Compute & Storage Constraints

### Accelerator Types

| Accelerator   | Hardware             | Notebook RAM | Notes                             |
| ------------- | -------------------- | ------------ | --------------------------------- |
| **GPU P100**  | 1× NVIDIA Tesla P100 | **~30 GB**   | Single GPU                        |
| **GPU T4 x2** | 2× NVIDIA T4         | **~30 GB**   | Dual GPU, same quota pool as P100 |
| **TPU v5e-8** | 8 TPU v5e cores      | **~330 GB**  | Very high RAM TPU VM              |

---

### Weekly Accelerator Quotas

| Resource | Weekly Limit |
|----------|---------------|
| **GPU (P100 + T4x2 combined)** | ~30 hours / week |
| **TPU** | ~20 hours / week |

> All GPU types share the same 30-hour pool.

---

### Maximum Runtime Per Session

| Mode                          | Listed Max Continuous Runtime |
| ----------------------------- | ----------------------------- |
| **CPU Only (No accelerator)** | ~**12 hours**                 |
| **GPU Notebook**              | **~12 hours**                 |
| **TPU Notebook**              | **~9 hours**                  |

---

### RAM Available

| Mode | RAM |
|------|-----|
| **CPU Only** | **~30 GB** |
| **GPU (P100 / T4x2)** | **~30 GB** |
| **TPU v5e-8** | **~330 GB** |

---

### Disk & Write Limits

| Mode         | Writable Disk (listed) | Practical Upper Limit | `/kaggle/working` | /kaggle/temp |
| ------------ | ---------------------- | --------------------- | ----------------- | ------------ |
| **CPU Only** | ~57 GB                 | ~100 GB               | **20 GB**         |              |
| **GPU**      | ~57 GB                 | ~100 GB               | **20 GB**         |              |
| **TPU**      | ~40 GB                 | ~100 GB               | **20 GB**         |              |

### Important Behavior
- Only files inside **`/kaggle/working` (20 GB)** are saved after commit.
- You can temporarily write much more data during the session (close to ~100 GB), but it is deleted when the session ends unless moved to `/kaggle/working`.

## 2. Connecting a GitHub Repository as a Dataset and Calling Scripts from It

- In the Kaggle notebook interface, in the Input Sectionclick on **Upload → New Dataset**.
- Choose the option to upload from a **Link and Import GitHub repository** and provide the repository URL(must be public repository).
- Kaggle will package the repository contents as a dataset, ask you to name it and attach it to your account.
- Once attached, the repository will be available to the notebook as an **input dataset**, and all scripts and files can be accessed directly from the mounted input directory.
- You can then import modules or execute scripts from this dataset just like local files if it has been added to input.
- This is exactly how the **[Baseline Notebook](https://www.kaggle.com/code/siddharthandileep/baseline-run-aisehack-test/)** posted as reference works with **[Baseline Code Github Repo](https://github.com/vaasew/baseline_anrf)**

- If changes are made to the remote GitHub repository and you want them to be reflected in Kaggle:
  - Navigate to the dataset’s page on Kaggle.
  - Use the Update dataset to pull the latest version from the GitHub repository.(You can even set periodic auto-updates)
  - Return to the notebook and click **Check for updates** on it's input dataset to sync the latest changes.
- This workflow allows you to maintain a single source of truth in GitHub while iterating seamlessly within Kaggle notebooks.


## 3. Understanding the raw data

In [ ]:
data_files='/kaggle/input/competitions/aisehack-theme-2'

# loading different features, you can refer to Data section of the competition webpage for insight into what each feature signifies
cpm=np.load(os.path.join(data_files,'raw','APRIL_16','cpm25.npy')) # pm concentrations
t2=np.load(os.path.join(data_files,'raw','APRIL_16','t2.npy')) # temperature
time=np.load(os.path.join(data_files,'raw','APRIL_16','time.npy')) # timestamp
rain=np.load(os.path.join(data_files,'raw','APRIL_16','rain.npy')) # rain

print(time[0])
# time is UTC

In [ ]:
print(time[-1])

print(cpm.shape)
print(t2.shape)
print(rain.shape)

#shape => (hour,latitudes,longitudes)
#you can view it as an hourly image of the subcontinent
#all features are aligned in time and space

In [ ]:
import seaborn as sns
sns.heatmap(cpm[0,::-1,:])
print(time[0]) #time stamp for the current image/pm2.5 distribution
#reversing dim 1 to view the sub continent top to bottom
#you should be able to notice the peninsular boundary

In [ ]:
sns.heatmap(cpm[45,::-1,:])
print(time[45]) #time stamp for the current image/pm2.5 distribution

# you can notice a sort of an episode happening in the extreme north

In [ ]:
sns.heatmap(cpm[6,::-1,:])
print(time[6])

In [ ]:
sns.heatmap(t2[60,::-1,:])
print(time[60])

### you can view most features provided like this, a few of them can be a lot sparser (like rain,some emission variables etc) so maybe they wont appeal to the visual cortex. Doesn't mean you should ignore them😞

In [ ]:
"""
you can notice how the feature rain for this hour is a lot sparser showing non zero values
for mostly just Northeastern India
"""
sns.heatmap(rain[6,::-1,:])
print(time[6])

## 4. Making train/val timeseries samples from the raw data


In [ ]:

# creating time series samples from the raw data
l=[]
stride=1
total_sample_hours=26
for i in range(0,cpm.shape[0]-total_sample_hours+1,stride):
    l.append(cpm[i:i+26,:,:])
cpm_samples=np.stack(l)
cpm_samples.shape

### Moving Window Sampling — Effect of Stride

**Time series (T = 12), window = 5**

Index:   01 02 03 04 05 06 07 08 09 10 11 12  
Data:    a  b  c  d  e  f  g  h  i  j  k  l

---

**Stride = 1 (maximum overlap)**

[ a  b  c  d  e ]  
&nbsp;&nbsp;[ b  c  d  e  f ]  
&nbsp;&nbsp;&nbsp;&nbsp;[ c  d  e  f  g ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ d  e  f  g  h ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ e  f  g  h  i ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ f  g  h  i  j ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ g  h  i  j  k ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ h  i  j  k  l ]

---

**Stride = 2 (moderate overlap)**

[ a  b  c  d  e ]  
&nbsp;&nbsp;&nbsp;&nbsp;[ c  d  e  f  g ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ e  f  g  h  i ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ g  h  i  j  k ]

---

**Stride = 3 (less overlap)**

[ a  b  c  d  e ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ d  e  f  g  h ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ g  h  i  j  k ]

---

**Stride = 5 (no overlap)**

[ a  b  c  d  e ]  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[ f  g  h  i  j ]

---

**Points to note**

- num_samples = floor((T - window_size) / stride) + 1

- The number of train/val samples you can make will be dependent on the stride and is honestly a hyperparameter.

- If you make stride=1 train/val samples for all features in float32 and write it to disk(like in `/kaggle/temp/`), it will take up ~76GB.

In [ ]:
l=[]
stride=1
total_sample_hours=26
for i in range(0,t2.shape[0]-total_sample_hours+1,stride):
    l.append(t2[i:i+26,:,:])
t2_samples=np.stack(l)
t2_samples.shape

print(np.stack([cpm_samples,t2_samples],axis=-1).shape)

#### you can imagine this stacked data as 690 video samples, each consisting of 26 time-ordered frames, where every frame is an image of size 140 × 124 with 2 channels.

## 5. Test inputs

In [ ]:
test_cpm=np.load(os.path.join(data_files,'test_in','cpm25.npy'))
test_cpm.shape

#### you are required to predict the next 16 hours of the `cpm25` feature for each of the 996 samples

### Please remember - 
- final prediction out shape => (996, 140, 124, 16)
- name it `preds.npy`
- store in `/kaggle/working/`

## 6. Full Notebook Pipeline


    A[Raw Data] --> B[Select Feature Subset]
    B --> C[Prepare Train/Val samples]
    C --> D[Dataset & DataLoader]
    D --> E[Model Training]
    E --> F[Best Checkpoint]

    G[test_in] --> H[For Same Feature Subset]
    H --> I[Load Best Checkpoint]
    I --> J[Batch Inference]
    J --> K[Save /kaggle/working/preds.npy]

### Please refer to **[Baseline Code Github Repo](https://github.com/vaasew/baseline_anrf)** and its README to see a working implementation of this pipeline.

## 7. Training Tips

### a. Modelling
- Operator-learning–based models have consistently demonstrated strong performance on this dataset. In particular, spectral operator learning architectures such as Fourier Neural Operators (FNO), Complex Neural Operators (CONO), aFNO, WNO and related variants are well suited for capturing complex spatio-temporal dynamics. Hence we recommend using them.
- The provided baseline architecture is composed of a version of stacked FNO2D blocks. A detailed description of the FNO architecture, its theoretical foundations, and the official reference implementation can be found in this paper: [Link](https://arxiv.org/pdf/2512.01421)
- Participants are encouraged to explore architectural improvements such as feature mixing strategies, skip or residual connections, warm up strategies etc.

### b. Dataloader
- The current DataLoader implementation in the baseline code introduces significant I/O overhead, especially when training with a large number of input features.
- In practice, this data loading can dominate the training loop, taking more than 4× time per epoch than the forward and backward passes combined.
- Participants are strongly advised to re-engineer the data pipeline to make it more time efficient.

### c. Pre-Processing
- Feature selection is critical for both model performance and computational efficiency.
- Participants are encouraged to run small-scale ablation experiments on subsets of input features to evaluate their individual and combined contributions.
- Identifying and removing redundant or low-impact features can lead to improved generalization, reduced memory usage, and faster training.
- Instead of relying only on global feature-wise min–max normalization, consider applying grid-wise normalization (per grid point for each feature) and using more outlier-robust scaling methods to better handle spatial variability and extreme pollution events.

### d. Training and Compute
- The baseline code using all 16 features takes approximately 9 hours to run end-to-end on an NVIDIA P100 GPU, consuming nearly one-third of a user’s weekly Kaggle GPU quota.
- This runtime should be treated as an upper bound rather than a target. If participants plan to experiment with more complex models or train for a larger number of epochs, it becomes pivotal to significantly reduce per-epoch and end-to-end training time to remain within practical compute limits. Participants are therefore expected to actively optimize training efficiency before scaling model complexity or training duration.
- **Team-based participation is strongly encouraged**, as larger teams effectively increase the total available Kaggle compute, enabling more extensive experimentation and faster iteration.
- Careful tuning of optimization hyperparameters, attention to memory usage, and consistent evaluation and reproducibility practices are important for achieving stable and comparable results.
